# Notebook 01 — Data Preparation & Exploratory Data Analysis
**Project**: Automated Customer Review Sentiment Analysis  
**Dataset**: Amazon Reviews 2023 (UCSD / McAuley Lab)  
**Author**: Sebastian López  
**Course**: Ironhack Data Science Bootcamp

---

## Overview

In this notebook, I prepare and explore the Amazon Reviews 2023 dataset for a multi-class sentiment classification task. My goal is to transform raw JSONL.gz review data into clean, balanced, and properly split datasets saved in HuggingFace Arrow format — ready for fine-tuning DistilBERT (Notebook 02) and RoBERTa (Notebook 03).

### Sentiment Label Mapping

| Star Rating | Sentiment Class | Label |
|-------------|-----------------|-------|
| 1–2 stars   | Negative        | 0     |
| 3 stars     | Neutral         | 1     |
| 4–5 stars   | Positive        | 2     |

### Notebook Structure
- **Section 0** — Setup & Environment  
- **Section 1** — Data Loading (streaming from HuggingFace Hub)  
- **Section 2** — Exploratory Data Analysis  
- **Section 3** — Text Preprocessing  
- **Section 4** — Dataset Balancing & Train/Val/Test Splitting  
- **Section 5** — Save to HuggingFace Arrow Format

---
## Section 0 — Setup & Environment

In this section, I mount Google Drive, install the required libraries, import all dependencies, and define the base paths used throughout the notebook. Running this cell first ensures the environment is consistent whether I'm working locally or on Google Colab.

In [ ]:
# ── Google Drive Mount ────────────────────────────────────────────────────────
# I mount Google Drive so all artifacts (datasets, images, CSVs) persist
# between Colab sessions. Without this, files would be lost on runtime restart.
import os

# Detect whether we are running inside Google Colab
try:
    import google.colab          # noqa: F401
    IN_COLAB = True              # Running on Colab
except ImportError:
    IN_COLAB = False             # Running locally

if IN_COLAB:
    # Mount Google Drive at /content/drive — a standard Colab pattern
    from google.colab import drive
    drive.mount('/content/drive')  # Expected output: 'Mounted at /content/drive'
    print("✅ Google Drive mounted successfully.")
else:
    # Running locally — no mount needed
    print("ℹ️  Not running in Colab — skipping Drive mount.")

In [ ]:
# ── Library Installation ──────────────────────────────────────────────────────
# I install all required packages in one shot to avoid version conflicts.
# Using --quiet suppresses verbose output and keeps the cell clean.
# Expected output: pip install success messages (or 'already satisfied').
!pip install --quiet \
    datasets \
    transformers \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    scikit-learn \
    tqdm

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
# Standard library
import re           # Regular expressions for text cleaning
import os           # OS-level path operations
import json         # JSON parsing (used for metadata inspection)
import warnings     # Suppress non-critical warnings

# Data manipulation
import pandas as pd               # DataFrame operations
import numpy as np                # Numerical operations

# Visualization
import matplotlib.pyplot as plt   # Base plotting
import matplotlib.ticker as mticker
import seaborn as sns             # Statistical visualization

# HuggingFace datasets library — used to stream and store data efficiently
from datasets import load_dataset, Dataset, DatasetDict

# Scikit-learn — used for stratified splitting and label analysis
from sklearn.model_selection import train_test_split

# Progress bars for long-running loops
from tqdm.auto import tqdm

# ── Global Configuration ──────────────────────────────────────────────────────
# Suppress FutureWarnings from HuggingFace / pandas to keep output clean
warnings.filterwarnings('ignore')

# Set a consistent random seed so results are reproducible across runs
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Apply a clean matplotlib style — 'seaborn-v0_8' is matplotlib ≥3.6 compatible
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ All imports successful.")

In [ ]:
# ── Path Configuration ────────────────────────────────────────────────────────
# I centralise all directory paths here so changing the Drive folder later
# only requires editing this one cell — nothing else in the notebook breaks.

if IN_COLAB:
    # BASE_DIR points to the project folder inside Google Drive
    BASE_DIR = '/content/drive/MyDrive/nlp-project/business-case-01'
else:
    # When running locally, use the current repository root
    BASE_DIR = os.getcwd()  # Resolves to repo root

# OUTPUT_DIR is where all generated artifacts are saved:
# processed datasets, plot PNGs, and summary CSVs
OUTPUT_DIR = os.path.join(BASE_DIR, 'data')

# Sub-directory for the HuggingFace Arrow dataset (consumed by NB02 & NB03)
DATASET_DIR = os.path.join(OUTPUT_DIR, 'dataset')

# Sub-directory for saved plot images
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')

# Create directories if they don't already exist
# exist_ok=True prevents errors if the folder was already created previously
os.makedirs(DATASET_DIR, exist_ok=True)  # Expected: no error, directory created/exists
os.makedirs(PLOTS_DIR, exist_ok=True)    # Expected: no error, directory created/exists

print(f"✅ Paths configured.")
print(f"   BASE_DIR    : {BASE_DIR}")
print(f"   OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"   DATASET_DIR : {DATASET_DIR}")
print(f"   PLOTS_DIR   : {PLOTS_DIR}")

---
## Section 1 — Data Loading

In this section, I load a large and diverse sample of the Amazon Reviews 2023 dataset. After the initial prototype (5 000 reviews per category), I decided to scale up to **all 33 product categories** to maximise representativeness and linguistic diversity.

**Decision rationale (2025-05-02):**
- **33 categories** (all of them, including Unknown and niche categories) ensure the model sees the full vocabulary spectrum — from technical automotive reviews to emotional pet supply testimonials.
- **30 000 reviews per category** (~990 000 total raw) was chosen over 50 000 to keep training time manageable while still being 20× larger than the prototype.
- **No MAX_PER_CLASS cap** — I let the minority class (Negatives, ~10-15 % of reviews) dictate the final balanced size. Capping would discard data we already spent time streaming and preprocessing.
- **Colab Pro** (~25 GB RAM) makes this feasible. The streaming pipeline (`streaming=True`) reads lazily from HuggingFace Hub, so we never download the full 750 GB.

My strategy:
1. Stream all 33 categories using `streaming=True` — lazy loading avoids downloading 750 GB upfront.
2. Take 30 000 samples per category to keep training time reasonable (~1 000 000 raw → ~400 000–500 000 balanced).
3. Accumulate per-category in RAM (30 000 Python dicts ≈ 90 MB per category) and convert to DataFrame at the end.

In [ ]:
# ── Category Selection — ALL 33 Categories ────────────────────────────────────
# I include every category in the Amazon Reviews 2023 dataset to maximise
# linguistic diversity and prevent domain-specific overfitting. Even niche
# categories like Subscription_Boxes (16K total) and Digital_Music (130K)
# contribute unique vocabulary patterns.
SELECTED_CATEGORIES = [
    'All_Beauty',
    'Amazon_Fashion',
    'Appliances',
    'Arts_Crafts_and_Sewing',
    'Automotive',
    'Baby_Products',
    'Beauty_and_Personal_Care',
    'Books',
    'CDs_and_Vinyl',
    'Cell_Phones_and_Accessories',
    'Clothing_Shoes_and_Jewelry',
    'Digital_Music',
    'Electronics',
    'Gift_Cards',
    'Grocery_and_Gourmet_Food',
    'Handmade_Products',
    'Health_and_Household',
    'Health_and_Personal_Care',
    'Home_and_Kitchen',
    'Industrial_and_Scientific',
    'Kindle_Store',
    'Magazine_Subscriptions',
    'Movies_and_TV',
    'Musical_Instruments',
    'Office_Products',
    'Patio_Lawn_and_Garden',
    'Pet_Supplies',
    'Software',
    'Sports_and_Outdoors',
    'Subscription_Boxes',    # Only 16K total — will take all of them
    'Tools_and_Home_Improvement',
    'Toys_and_Games',
    'Unknown',               # 63.8M reviews — no category label, but valuable for sentiment
]

# Number of reviews to sample per category.
# At 30,000 samples × 33 categories ≈ 990,000 total reviews.
# Chosen over 50K to keep training time manageable (Colab Pro with T4 GPU).
N_SAMPLES_PER_CATEGORY = 30_000

# HuggingFace dataset identifier for Amazon Reviews 2023
# See: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023
HF_DATASET_ID = 'McAuley-Lab/Amazon-Reviews-2023'

print(f"✅ Configuration set.")
print(f"   Categories  : {len(SELECTED_CATEGORIES)}")
print(f"   Samples/cat : {N_SAMPLES_PER_CATEGORY:,}")
print(f"   Max total   : {len(SELECTED_CATEGORIES) * N_SAMPLES_PER_CATEGORY:,} reviews")

In [ ]:
# ── Stream & Sample Reviews ───────────────────────────────────────────────────
# I iterate over each selected category, stream its reviews, and collect
# N_SAMPLES_PER_CATEGORY rows into a list. Streaming means HuggingFace sends
# data in chunks — we never load the full category into RAM at once.
#
# Expected output: progress bars per category, then a summary DataFrame shape.

all_records = []  # Accumulator list — holds dicts from all categories

for category in SELECTED_CATEGORIES:
    print(f"\n📦 Loading category: {category}")

    # load_dataset with streaming=True returns an IterableDataset.
    # The config name follows the pattern 'raw_review_{category}'.
    # trust_remote_code=True is required by this dataset's loading script.
    try:
        streamed = load_dataset(
            HF_DATASET_ID,
            f'raw_review_{category}',  # Dataset config name for this category
            split='full',              # 'full' is the only split in this dataset
            streaming=True,            # Lazy loading — avoids downloading everything
            trust_remote_code=True     # Required by the dataset's custom loading script
        )
    except Exception as e:
        # If a category fails (e.g., network issue), skip it and continue
        print(f"  ⚠️  Skipping {category} — {e}")
        continue

    # Sample N rows using tqdm for a visible progress bar.
    # iter() + next() pattern works on IterableDataset (no random access).
    category_records = []  # Temp list for this category's samples
    for record in tqdm(streamed, total=N_SAMPLES_PER_CATEGORY, desc=f"  {category}", leave=False):
        # Add the category name as a new field so we can track it later
        record['category'] = category
        category_records.append(record)  # Append the enriched record

        # Stop once we've collected the target number of samples
        if len(category_records) >= N_SAMPLES_PER_CATEGORY:
            break

    # Merge this category's records into the global accumulator
    all_records.extend(category_records)
    print(f"  ✅ Collected {len(category_records):,} records from {category}")

# Convert the flat list of dicts into a pandas DataFrame for easy manipulation
# Expected: df with ~990,000 rows and multiple columns
df_raw = pd.DataFrame(all_records)

print(f"\n✅ Raw DataFrame shape: {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")

In [ ]:
# ── Inspect Raw Data ──────────────────────────────────────────────────────────
# I display the first few rows and data types to understand the structure
# before doing any cleaning. This is always my first step after loading.
#
# Expected output: a table showing columns like rating, text, title, asin, etc.

print("=" * 60)
print("RAW DATA — FIRST 5 ROWS")
print("=" * 60)
# Display key columns only to keep the output readable
display(df_raw[['rating', 'text', 'title', 'asin', 'user_id', 'category']].head())

print("\n" + "=" * 60)
print("DATA TYPES & NON-NULL COUNTS")
print("=" * 60)
# df.info() shows column dtypes and how many non-null values exist in each column
df_raw.info()

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS (numeric columns)")
print("=" * 60)
# describe() gives count, mean, std, min, quartiles, max for numeric columns
display(df_raw.describe())

---
## Section 2 — Exploratory Data Analysis (EDA)

In this section, I explore the raw dataset to understand its structure, identify quality issues, and build intuition for the preprocessing steps I'll take in Section 3. I look at:

- **Rating distribution** — Are reviews biased toward positive ratings?
- **Text length distribution** — How long are reviews? Are there outliers?
- **Category distribution** — Is the dataset evenly distributed across categories?
- **Missing values** — Which columns have gaps that need handling?
- **Sample reviews per sentiment class** — What does each class look and feel like?

In [ ]:
# ── Plot 1: Rating Distribution ───────────────────────────────────────────────
# I plot the count of reviews per star rating (1–5) to see if the dataset
# is skewed. I check whether the dataset shows the positivity bias commonly reported for Amazon reviews (majority 4–5 star ratings).
#
# Expected output: bar chart saved to PLOTS_DIR + printed value counts.

# Count reviews per rating value
rating_counts = df_raw['rating'].value_counts().sort_index()  # Sort by rating (1→5)

print("Rating value counts:")
print(rating_counts)
print(f"\nMean rating  : {df_raw['rating'].mean():.2f}")
print(f"Median rating: {df_raw['rating'].median():.1f}")

# Create figure
fig, ax = plt.subplots(figsize=(8, 5))  # 8×5 inches gives a clean readable chart

# Bar chart — each bar represents one star rating
bars = ax.bar(
    rating_counts.index,        # X axis: star ratings 1–5
    rating_counts.values,       # Y axis: count of reviews per rating
    color=sns.color_palette('Blues', 5),  # Blue gradient from light (1★) to dark (5★)
    edgecolor='white',          # White border between bars for clarity
    linewidth=0.8
)

# Annotate each bar with its count so exact values are visible
for bar, count in zip(bars, rating_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # Horizontal center of bar
        bar.get_height() + 100,              # Slightly above the bar top
        f'{count:,}',                        # Formatted with thousands separator
        ha='center', va='bottom', fontsize=10
    )

# Labels and formatting
ax.set_title('Rating Distribution — Raw Dataset', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Star Rating', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_xticks([1, 2, 3, 4, 5])  # Ensure only integer ticks on X axis
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()  # Prevent labels from being clipped

# Save plot to disk before displaying
plot_path = os.path.join(PLOTS_DIR, 'rating_distribution.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')  # dpi=150 gives good quality
print(f"\n💾 Plot saved → {plot_path}")

plt.show()  # Display inline in the notebook

In [ ]:
# ── Plot 2: Review Text Length Distribution ───────────────────────────────────
# I compute the character length of each review text and plot its distribution.
# This helps me decide a sensible minimum length filter and understand
# how much text the models will receive as input.
#
# Expected output: histogram with percentile annotations.

# Compute character count for each review (NaN text → 0 characters)
# fillna('') prevents errors on rows where text is missing
df_raw['text_length'] = df_raw['text'].fillna('').apply(len)

# Compute key percentiles to understand the distribution
p10  = np.percentile(df_raw['text_length'], 10)    # 10th percentile
p50  = np.percentile(df_raw['text_length'], 50)    # Median (50th)
p90  = np.percentile(df_raw['text_length'], 90)    # 90th percentile
p99  = np.percentile(df_raw['text_length'], 99)    # 99th — outlier threshold

print(f"Text length stats:")
print(f"  Min      : {df_raw['text_length'].min()}")
print(f"  P10      : {p10:.0f}")
print(f"  Median   : {p50:.0f}")
print(f"  Mean     : {df_raw['text_length'].mean():.0f}")
print(f"  P90      : {p90:.0f}")
print(f"  P99      : {p99:.0f}")
print(f"  Max      : {df_raw['text_length'].max()}")

fig, ax = plt.subplots(figsize=(10, 5))

# Plot histogram clipped at the 99th percentile to avoid outlier distortion
ax.hist(
    df_raw['text_length'].clip(upper=p99),  # Cap extreme values for readability
    bins=60,                                 # 60 bins gives good granularity
    color='#0891B2',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.85
)

# Draw vertical lines at key percentiles so the distribution is interpretable
for pct, label, color in [
    (p10, 'P10', '#EA580C'),
    (p50, 'P50 (median)', '#DC2626'),
    (p90, 'P90', '#0891B2'),
]:
    ax.axvline(pct, color=color, linestyle='--', linewidth=1.4, label=f'{label}: {pct:.0f} chars')

ax.set_title('Review Text Length Distribution (clipped at P99)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Characters', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'text_length_distribution.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Plot saved → {plot_path}")
plt.show()

In [ ]:
# ── Plot 3: Reviews per Category ──────────────────────────────────────────────
# I check whether the sampling produced an evenly distributed dataset.
# Small deviations are expected if a category had fewer than N_SAMPLES_PER_CATEGORY
# reviews available.
#
# Expected output: horizontal bar chart showing counts per category.

# Count reviews per category, sort descending so the longest bar is at top
category_counts = df_raw['category'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

# Horizontal bar chart — easier to read when category names are long
bars = ax.barh(
    category_counts.index,      # Y axis: category names
    category_counts.values,     # X axis: review counts
    color=sns.color_palette('husl', len(category_counts)),  # Distinct colour per category (33 cats)
    edgecolor='white'
)

# Annotate each bar with its exact count
for bar, count in zip(bars, category_counts.values):
    ax.text(
        bar.get_width() + 30,   # Slightly to the right of the bar end
        bar.get_y() + bar.get_height() / 2,  # Vertical center of bar
        f'{count:,}',
        va='center', fontsize=9
    )

ax.set_title('Number of Reviews per Category', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'category_distribution.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"💾 Plot saved → {plot_path}")
plt.show()

In [ ]:
# ── Missing Values Analysis ───────────────────────────────────────────────────
# I count missing values per column to understand data completeness.
# The critical columns for this project are 'rating' and 'text' —
# rows missing either of these will be dropped in Section 3.
#
# Expected output: table of missing counts and percentages per column.

# Compute absolute count and percentage of missing values per column
missing_counts = df_raw.isnull().sum()          # Count of NaN per column
missing_pct    = (missing_counts / len(df_raw) * 100).round(2)  # As percentage

# Combine into a single summary DataFrame for readability
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %'    : missing_pct
}).sort_values('Missing Count', ascending=False)  # Sort by worst first

print("Missing Values Summary:")
display(missing_summary)

# Also check for empty strings in the 'text' column (not caught by isnull)
# An empty string is just as useless as NaN for NLP purposes
n_empty_text = (df_raw['text'].fillna('').str.strip() == '').sum()
print(f"\nRows with empty string in 'text': {n_empty_text:,}")

In [ ]:
# ── Sample Reviews per Sentiment Class ───────────────────────────────────────
# Before preprocessing, I apply the rating → sentiment label mapping temporarily
# and display example reviews from each class. This builds intuition for what
# the model might learn to distinguish across classes.
#
# Mapping: 1-2★ → Negative (0), 3★ → Neutral (1), 4-5★ → Positive (2)
#
# Expected output: printed examples for each sentiment class.

def rating_to_label(rating):
    """Map a star rating (1–5) to a sentiment label (0, 1, or 2).
    Returns:
        0 for Negative (1–2 stars)
        1 for Neutral  (3 stars)
        2 for Positive (4–5 stars)
    """
    if rating <= 2:
        return 0   # Negative
    elif rating == 3:
        return 1   # Neutral
    else:
        return 2   # Positive

# Apply the mapping to create a temporary label column for preview purposes
df_raw['temp_label'] = df_raw['rating'].apply(rating_to_label)

# Map integer labels to descriptive names for printing
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

# Show 2 sample reviews per sentiment class
print("=" * 70)
print("SAMPLE REVIEWS PER SENTIMENT CLASS")
print("=" * 70)

for label_id in [0, 1, 2]:  # Iterate over Negative, Neutral, Positive
    # Filter rows belonging to this class where text is non-null
    class_df = df_raw[
        (df_raw['temp_label'] == label_id) &
        (df_raw['text'].notna())
    ]

    # Sample 2 rows randomly (random_state=RANDOM_SEED for reproducibility)
    samples = class_df.sample(min(2, len(class_df)), random_state=RANDOM_SEED)

    print(f"\n{'─'*70}")
    print(f"  CLASS {label_id} — {label_names[label_id].upper()}")
    print(f"{'─'*70}")

    for _, row in samples.iterrows():
        # Truncate long reviews to 300 characters for display
        preview = str(row['text'])[:300].replace('\n', ' ')
        print(f"  ★ {row['rating']:.0f} | {row['category']}")
        print(f"  \"{preview}...\"")
        print()

# Remove the temporary label column — it will be properly recreated in Section 3
df_raw.drop(columns=['temp_label'], inplace=True)

In [ ]:
# ── Key Summary Statistics ────────────────────────────────────────────────────
# I compile a quick reference table of the most important dataset-level numbers.
# This gives a snapshot I can reference throughout the project.
#
# Expected output: printed summary block.

print("=" * 50)
print("  EDA SUMMARY — RAW DATASET")
print("=" * 50)

# Total reviews in the raw DataFrame
print(f"  Total reviews       : {len(df_raw):,}")

# Number of unique product categories in the loaded data
print(f"  Unique categories   : {df_raw['category'].nunique()}")

# Unique reviewers (user_id)
print(f"  Unique users        : {df_raw['user_id'].nunique():,}")

# Unique products (parent_asin)
if 'parent_asin' in df_raw.columns:
    print(f"  Unique products     : {df_raw['parent_asin'].nunique():,}")

# Mean and median star rating across all loaded reviews
print(f"  Mean star rating    : {df_raw['rating'].mean():.2f}")
print(f"  Median star rating  : {df_raw['rating'].median():.1f}")

# Average text length in characters (excluding NaN)
avg_len = df_raw['text'].dropna().apply(len).mean()
print(f"  Avg text length     : {avg_len:.0f} chars")

# Count of rows with missing text (need to drop in preprocessing)
n_missing_text = df_raw['text'].isna().sum()
print(f"  Missing text rows   : {n_missing_text:,} ({n_missing_text/len(df_raw)*100:.1f}%)")

print("=" * 50)

### Extended EDA — NLP-Specific Analysis

Beyond the basic numerical distributions above, I now explore analyses that are specifically relevant for Natural Language Processing. These analyses examine whether text-level patterns support the use of a transformer model over a simpler baseline — they explore whether patterns exist in the **text itself**, beyond what the rating distribution shows.

I focus on:
- **Word-level patterns** — vocabulary differences between sentiment classes
- **Text structure** — do negative reviews differ in length from positive ones?
- **Metadata signals** — verified purchases, helpful votes, and what they reveal
- **Category insights** — does sentiment vary across product types?

These analyses inform preprocessing decisions (token limits, text filtering) and provide storytelling material for the final report.

In [ ]:
# ── Plot 7: Word Count Distribution ───────────────────────────────────────────
# Character count shows the volume of raw text. Word count estimates how much
# *language* the model actually needs to process. DistilBERT and RoBERTa have
# a 512-token limit (~394 words for English). I identify how many reviews would
# be affected by truncation, to understand the potential information loss.
#
# Expected output: histogram with token-limit reference line.

# Compute word count per review (splitting on whitespace)
df_raw['word_count'] = df_raw['text'].fillna('').apply(lambda x: len(str(x).split()))

# Key percentiles for the distribution
wp10 = np.percentile(df_raw['word_count'], 10)
wp50 = np.percentile(df_raw['word_count'], 50)
wp90 = np.percentile(df_raw['word_count'], 90)
wp99 = np.percentile(df_raw['word_count'], 99)

# Rough estimate: English averages ~1.3 tokens per word.
# Reviews above ~394 words are likely to exceed 512 tokens.
est_token_limit_words = 512 / 1.3

n_above_token_limit = (df_raw['word_count'] > est_token_limit_words).sum()
pct_truncated = n_above_token_limit / len(df_raw) * 100

print("Word count statistics:")
print(f"  Min       : {df_raw['word_count'].min()}")
print(f"  P10       : {wp10:.0f}")
print(f"  Median    : {wp50:.0f}")
print(f"  Mean      : {df_raw['word_count'].mean():.0f}")
print(f"  P90       : {wp90:.0f}")
print(f"  P99       : {wp99:.0f}")
print(f"  Max       : {df_raw['word_count'].max()}")
print(f"\n  Reviews likely to be truncated at 512 tokens (~{est_token_limit_words:.0f} words):")
print(f"  → {n_above_token_limit:,} reviews ({pct_truncated:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    df_raw['word_count'].clip(upper=wp99),  # Clip outliers for readability
    bins=60,
    color='#0891B2',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.85
)

# Percentile reference lines
for pct, label, color in [
    (wp10, 'P10', '#EA580C'),
    (wp50, 'P50 (median)', '#DC2626'),
    (wp90, 'P90', '#0891B2'),
]:
    ax.axvline(pct, color=color, linestyle='--', linewidth=1.4,
               label=f'{label}: {pct:.0f} words')

# 512-token reference line
ax.axvline(est_token_limit_words, color='#DC2626', linestyle=':', linewidth=2,
           label=f'~512 tokens ≈ {est_token_limit_words:.0f} words')

ax.set_title('Review Word Count Distribution (clipped at P99)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Words', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'word_count_distribution.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Plot saved → {plot_path}")
plt.show()

In [ ]:
# ── Plot 8: Text Length by Sentiment Class ────────────────────────────────────
# Previous research on online reviews documents a *negativity bias* — users may
# invest more words describing negative experiences than positive ones. I test
# whether this pattern holds in the Amazon dataset.
#
# Expected output: three side-by-side boxplots with descriptive stats.

# Re-create sentiment labels for this analysis only
# rating_to_label() was defined in Section 2 (cell-sample-reviews)
df_raw['eda_label'] = df_raw['rating'].apply(rating_to_label)

label_display = ['Negative\n(1-2★)', 'Neutral\n(3★)', 'Positive\n(4-5★)']
box_colors    = ['#DC2626', '#EA580C', '#0891B2']

# Prepare data: list of arrays, one per class
data_by_class = [
    df_raw[df_raw['eda_label'] == i]['text_length'].dropna().values
    for i in [0, 1, 2]
]

fig, ax = plt.subplots(figsize=(9, 5.5))

bp = ax.boxplot(
    data_by_class,
    labels=label_display,
    patch_artist=True,
    widths=0.5,
    showfliers=True,
    showmeans=True,
    meanprops=dict(marker='D', markerfacecolor='#334155', markersize=6)
)

# Colour each box by sentiment
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Print per-class summary
print("Text length by sentiment class:")
print("─" * 65)
print(f"  {'Class':<18s} {'Mean':>7s}  {'Median':>7s}  {'Std':>7s}  {'Count':>8s}")
print("─" * 65)
for i, label_id in enumerate([0, 1, 2]):
    lengths = df_raw[df_raw['eda_label'] == label_id]['text_length']
    name = label_display[i].replace('\n', ' ')
    print(f"  {name:<18s} {lengths.mean():>7.0f}  {lengths.median():>7.0f}  "
          f"{lengths.std():>7.0f}  {len(lengths):>8,}")

ax.set_title('Review Text Length by Sentiment Class', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Characters', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'text_length_by_sentiment.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Plot saved → {plot_path}")
plt.show()

# Clean up temporary label after the analysis
df_raw.drop(columns=['eda_label'], inplace=True)

In [ ]:
# ── Plot 9: Top 20 Most Frequent Words by Sentiment Class ────────────────────
# I extract the most common words per sentiment class after removing stopwords.
# This surfaces the vocabulary patterns that the model might leverage to
# distinguish sentiment classes. If word distributions differ meaningfully
# between classes, a transformer should be able to capture these differences.
#
# Expected output: three horizontal bar charts (one per sentiment class).

# Install NLTK for stopwords — only needed in this cell.
# (Placed here rather than in cell-pip-install because no other
#  notebook in the pipeline requires NLTK.)
!pip install --quiet nltk
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from collections import Counter

# ── Stopwords: NLTK standard + review-specific domain terms ───────────────
# I combine the NLTK English stopword list (179 words, academically
# standard and citable) with a curated set of review-domain terms that
# appear in virtually every review but carry no sentiment signal
# (e.g., 'product', 'bought', 'star'). Using NLTK alone would leave
# these high-frequency noise terms visible, diluting the insight from
# the frequency analysis.
STOPWORDS = set(stopwords.words('english')) | {
    # Review-specific noise terms — frequent but carry zero sentiment
    'product', 'item', 'bought', 'purchased',
    'review', 'reviews', 'star', 'stars', 'amazon',
    # Common contractions that NLTK does not cover
    'don', 'didn', 'doesn', 'wasn', 'weren',
    'isn', 'aren', 'couldn', 'wouldn', 'shouldn', 'haven',
    # Generic evaluative terms — appear in all sentiment classes
    'good', 'bad', 'great', 'use', 'used', 'using',
}

def get_top_words(text_series, n=20):
    """Extract top N most frequent words from a text Series.

    Applies minimal cleaning: lowercase, extract alphabetic tokens,
    filter by length ≥ 3, and remove stopwords.
    """
    all_words = []
    for text in text_series.dropna():
        # Extract only alphabetic sequences (no digits, no punctuation)
        words = re.findall(r'[a-z]+', str(text).lower())
        # Keep words with ≥3 chars that are not stopwords
        words = [w for w in words if len(w) >= 3 and w not in STOPWORDS]
        all_words.extend(words)
    return Counter(all_words).most_common(n)

# Compute and plot top words for each class
class_config = [
    (0, 'NEGATIVE (1-2★)', '#DC2626'),
    (1, 'NEUTRAL (3★)',    '#EA580C'),
    (2, 'POSITIVE (4-5★)', '#0891B2'),
]

df_raw['eda_label'] = df_raw['rating'].apply(rating_to_label)

for label_id, class_name, color in class_config:
    class_texts = df_raw[df_raw['eda_label'] == label_id]['text']
    top = get_top_words(class_texts)

    words, counts = zip(*top)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(
        list(reversed(words)),
        list(reversed(counts)),
        color=color,
        edgecolor='white',
        height=0.7
    )

    # Annotate each bar with its count
    max_count = max(counts)
    for bar, count in zip(bars, reversed(counts)):
        ax.text(
            bar.get_width() + max_count * 0.015,
            bar.get_y() + bar.get_height() / 2,
            f'{count:,}',
            va='center', fontsize=9
        )

    ax.set_title(f'Top 20 Words — {class_name} Reviews', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Frequency', fontsize=11)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f'top_words_class_{label_id}.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"💾 Plot saved → {plot_path}")
    plt.show()

df_raw.drop(columns=['eda_label'], inplace=True)

In [ ]:
# ── Verified Purchase vs Rating Distribution ───────────────────────────────────
# Amazon includes a 'verified_purchase' flag indicating whether the reviewer
# actually bought the product. I analyse whether verified buyers rate
# products differently — a signal worth investigating for potential review bias.
#
# Expected output: grouped bar chart + percentage comparison table.

if 'verified_purchase' in df_raw.columns:
    n_verified   = df_raw['verified_purchase'].sum()
    n_unverified = len(df_raw) - n_verified

    print("Verified Purchase Analysis:")
    print(f"  Verified purchases   : {n_verified:,} ({n_verified/len(df_raw)*100:.1f}%)")
    print(f"  Non-verified reviews : {n_unverified:,} ({n_unverified/len(df_raw)*100:.1f}%)")

    # Rating distribution (%) per group
    verified_dist   = df_raw[df_raw['verified_purchase'] == True]['rating'].value_counts(normalize=True).sort_index() * 100
    unverified_dist = df_raw[df_raw['verified_purchase'] == False]['rating'].value_counts(normalize=True).sort_index() * 100

    print("\nRating distribution (%):")
    print(f"  {'Rating':<8} {'Verified':>10} {'Unverified':>12}")
    print("  " + "─" * 30)
    for rating in sorted(df_raw['rating'].dropna().unique()):
        vp = verified_dist.get(rating, 0)
        up = unverified_dist.get(rating, 0)
        print(f"  {int(rating):<8} {vp:>9.1f}% {up:>11.1f}%")

    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(8, 5))

    x = np.arange(1, 6)   # Ratings 1-5
    width = 0.35

    verified_counts   = [df_raw[(df_raw['verified_purchase'] == True)  & (df_raw['rating'] == r)].shape[0] for r in range(1, 6)]
    unverified_counts = [df_raw[(df_raw['verified_purchase'] == False) & (df_raw['rating'] == r)].shape[0] for r in range(1, 6)]

    bars1 = ax.bar(x - width/2, verified_counts, width,
                   label='Verified Purchase', color='#0891B2', edgecolor='white', linewidth=0.6)
    bars2 = ax.bar(x + width/2, unverified_counts, width,
                   label='Not Verified', color='#EA580C', edgecolor='white', linewidth=0.6)

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., h + max(verified_counts + unverified_counts)*0.01,
                    f'{int(h):,}', ha='center', va='bottom', fontsize=8)

    ax.set_title('Rating Distribution: Verified vs Unverified Purchases', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Star Rating', fontsize=11)
    ax.set_ylabel('Number of Reviews', fontsize=11)
    ax.set_xticks(x)
    ax.legend(fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, 'verified_purchase_analysis.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n💾 Plot saved → {plot_path}")
    plt.show()
else:
    print("⚠️  Column 'verified_purchase' not found in the dataset. Skipping this analysis.")

In [ ]:
# ── Helpful Votes Analysis ────────────────────────────────────────────────────
# The 'helpful_vote' column counts how many users found a review useful.
# I test a common hypothesis from the review-helpfulness literature: do
# extreme ratings (1★ and 5★) receive more helpful votes? If so, this could
# indicate that strongly emotional reviews are perceived as more informative
# — a pattern worth documenting before model training.
#
# Expected output: boxplot of helpful votes by rating + aggregation table.

if 'helpful_vote' in df_raw.columns:
    n_helpful = (df_raw['helpful_vote'] > 0).sum()

    print("Helpful Votes Analysis:")
    print(f"  Reviews with ≥1 helpful vote : {n_helpful:,} ({n_helpful/len(df_raw)*100:.1f}%)")
    print(f"  Mean helpful votes            : {df_raw['helpful_vote'].mean():.1f}")
    print(f"  Median helpful votes          : {df_raw['helpful_vote'].median():.0f}")
    print(f"  Max helpful votes             : {df_raw['helpful_vote'].max():,}")

    # Mean helpful votes per star rating
    helpful_by_rating = df_raw.groupby('rating')['helpful_vote'].agg(
        Mean='mean', Total='sum', Count='count'
    ).round(1)
    print("\nHelpful votes by star rating:")
    print(helpful_by_rating.to_string())

    # Boxplot (outliers hidden for readability — max helpful_vote can be extreme)
    ratings = sorted(int(r) for r in df_raw['rating'].dropna().unique())
    data_by_rating = [df_raw[df_raw['rating'] == r]['helpful_vote'].values for r in ratings]

    fig, ax = plt.subplots(figsize=(9, 5))

    bp = ax.boxplot(
        data_by_rating,
        labels=[f'{r}★' for r in ratings],
        patch_artist=True,
        widths=0.5,
        showfliers=False  # Outliers distort the chart; stats table above has the max
    )

    colors_rating = sns.color_palette('Blues', len(ratings))
    for patch, color in zip(bp['boxes'], colors_rating):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    # Add mean annotation above each box
    for i, r in enumerate(ratings):
        mean_val = df_raw[df_raw['rating'] == r]['helpful_vote'].mean()
        ax.annotate(f'{mean_val:.1f}', xy=(i+1, mean_val),
                    xytext=(i+1, mean_val + df_raw['helpful_vote'].quantile(0.90)*0.15),
                    ha='center', fontsize=8, color='#334155', fontweight='bold')

    ax.set_title('Helpful Votes Distribution by Star Rating (outliers hidden)', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Star Rating', fontsize=12)
    ax.set_ylabel('Number of Helpful Votes', fontsize=12)

    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, 'helpful_votes_by_rating.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n💾 Plot saved → {plot_path}")
    plt.show()
else:
    print("⚠️  Column 'helpful_vote' not found in the dataset. Skipping this analysis.")

In [ ]:
# ── Sentiment Distribution per Category ────────────────────────────────────────
# I analyse how sentiment varies across product categories. Some categories
# (e.g., Books) tend to attract nuanced reviews; others (e.g., Gift Cards)
# are almost purely transactional. This helps assess whether a single model
# can generalise across all 33 categories.
#
# Expected output: grouped bar chart + printed summary for all categories.

df_raw['eda_label'] = df_raw['rating'].apply(rating_to_label)

# Compute % of each sentiment per category
sentiment_by_cat = (
    df_raw
    .groupby('category')['eda_label']
    .value_counts(normalize=True)
    .unstack(fill_value=0) * 100
)
# Sort by positive % ascending (most-critical categories first)
sentiment_by_cat = sentiment_by_cat.sort_values(by=2, ascending=True)

# Print per-category breakdown
print("Sentiment distribution per category (sorted by % Positive):")
print("─" * 70)
print(f"  {'Category':<35s} {'Neg %':>6s} {'Neu %':>6s} {'Pos %':>6s}  {'Total':>7s}")
print("─" * 70)
for cat in sentiment_by_cat.index:
    row = sentiment_by_cat.loc[cat]
    total = df_raw[df_raw['category'] == cat].shape[0]
    neg = row.get(0, 0)
    neu = row.get(1, 0)
    pos = row.get(2, 0)
    print(f"  {cat:<35s} {neg:>5.0f}% {neu:>5.0f}% {pos:>5.0f}%  {total:>7,}")

# Grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))

categories = sentiment_by_cat.index
x = np.arange(len(categories))
width = 0.25

neg_vals = sentiment_by_cat[0].values if 0 in sentiment_by_cat.columns else np.zeros(len(categories))
neu_vals = sentiment_by_cat[1].values if 1 in sentiment_by_cat.columns else np.zeros(len(categories))
pos_vals = sentiment_by_cat[2].values if 2 in sentiment_by_cat.columns else np.zeros(len(categories))

ax.bar(x - width, neg_vals, width, label='Negative', color='#DC2626', edgecolor='white', linewidth=0.3)
ax.bar(x,         neu_vals, width, label='Neutral',  color='#EA580C', edgecolor='white', linewidth=0.3)
ax.bar(x + width, pos_vals, width, label='Positive', color='#0891B2', edgecolor='white', linewidth=0.3)

ax.set_title('Sentiment Distribution by Product Category', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('% of Reviews', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=7)
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}%'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'sentiment_by_category.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Plot saved → {plot_path}")
plt.show()

df_raw.drop(columns=['eda_label'], inplace=True)

### Additional EDA — Underused Columns & Data Quality

So far I have focused on the `text` and `rating` columns. But the Amazon Reviews dataset includes several other fields that could reveal patterns. I now explore:

- **Title vs Body** — do review titles carry different signals than the body?
- **Extreme ratings** — what vocabulary is *unique* to 1★ and 5★ reviews?
- **Temporal trends** — how have review patterns changed over time?
- **Data quality** — are there duplicate or near-duplicate reviews?

These are small, focused investigations — each one either surfaces a finding or confirms an assumption. In the spirit of scientific EDA, I report what I find regardless of whether it confirms my expectations.

In [ ]:
# ── Title vs Body Length Comparison ───────────────────────────────────────────
# Review titles are typically short, emotionally charged summaries.
# I compare the character length of titles against the body text to see
# whether titles are disproportionately extreme (short and punchy)
# compared to the more measured body text.
#
# Expected output: side-by-side boxplots + descriptive stats.

if 'title' in df_raw.columns:
    # Compute title length (handle missing titles gracefully)
    df_raw['title_length'] = df_raw['title'].fillna('').apply(len)

    # Text length was already computed in Plot 2 (cell-text-length)

    print("Title vs Body Text — Length comparison:")
    print("─" * 50)
    print(f"  Title length — mean: {df_raw['title_length'].mean():.0f}  "
          f"median: {df_raw['title_length'].median():.0f}  "
          f"std: {df_raw['title_length'].std():.0f}")
    print(f"  Body length  — mean: {df_raw['text_length'].mean():.0f}  "
          f"median: {df_raw['text_length'].median():.0f}  "
          f"std: {df_raw['text_length'].std():.0f}")

    # Ratio: body / title — how many times longer is the body?
    # Avoid division by zero: filter titles with at least 1 char
    mask = df_raw['title_length'] > 0
    ratio = df_raw.loc[mask, 'text_length'] / df_raw.loc[mask, 'title_length']
    print(f"\n  Body/Title ratio  — mean: {ratio.mean():.1f}x  median: {ratio.median():.1f}x")

    # Boxplots side by side (log scale — body is orders of magnitude longer)
    fig, ax = plt.subplots(figsize=(8, 5))

    bp = ax.boxplot(
        [df_raw['title_length'].values, df_raw['text_length'].values],
        labels=['Title', 'Body (review text)'],
        patch_artist=True,
        widths=0.4,
        showfliers=True,
        showmeans=True,
        meanprops=dict(marker='D', markerfacecolor='#334155', markersize=6)
    )

    for patch, color in zip(bp['boxes'], ['#EA580C', '#0891B2']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_yscale('log')  # Log scale — body is much longer than title
    ax.set_title('Title vs Body Text Length (log scale)', fontsize=14, fontweight='bold', pad=12)
    ax.set_ylabel('Number of Characters (log scale)', fontsize=12)

    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, 'title_vs_body_length.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n💾 Plot saved → {plot_path}")
    plt.show()

    df_raw.drop(columns=['title_length'], inplace=True)
else:
    print("⚠️  Column 'title' not found in the dataset. Skipping this analysis.")

In [ ]:
# ── Extreme Ratings Word Comparison: 1★ vs 5★ ──────────────────────────────
# Beyond the top words per class, I compare vocabulary that is *unique*
# to the most extreme ratings. Words that appear in 1★ but rarely in 5★
# (and vice versa) might reveal the core drivers of satisfaction and
# frustration — insights that aggregate metrics like mean rating hide.
#
# NLTK stopwords were installed and downloaded in the Top Words cell (Plot 9).
#
# Expected output: two side-by-side bar charts of distinctive extreme-rating words.

from collections import Counter
from nltk.corpus import stopwords

# Reuse the same stopword set defined in Plot 9
STOPWORDS = set(stopwords.words('english')) | {
    'product', 'item', 'bought', 'purchased',
    'review', 'reviews', 'star', 'stars', 'amazon',
    'don', 'didn', 'doesn', 'wasn', 'weren',
    'isn', 'aren', 'couldn', 'wouldn', 'shouldn', 'haven',
    'good', 'bad', 'great', 'use', 'used', 'using',
}

def get_word_freqs(text_series, min_len=3):
    """Return Counter of word frequencies after stopword removal."""
    all_words = []
    for text in text_series.dropna():
        words = re.findall(r'[a-z]+', str(text).lower())
        words = [w for w in words if len(w) >= min_len and w not in STOPWORDS]
        all_words.extend(words)
    return Counter(all_words)

# Filter reviews with exactly 1★ and exactly 5★
texts_1star = df_raw[df_raw['rating'] == 1.0]['text']
texts_5star = df_raw[df_raw['rating'] == 5.0]['text']

print(f"1★ reviews: {len(texts_1star):,}  |  5★ reviews: {len(texts_5star):,}")

# Compute word frequencies
freq_1star = get_word_freqs(texts_1star)
freq_5star = get_word_freqs(texts_5star)

# Find words distinctive to each extreme
# A word is "distinctive" if its frequency ratio between classes is ≥ 2×
all_words_1 = set(freq_1star.keys())
all_words_5 = set(freq_5star.keys())
shared_words = all_words_1 & all_words_5

# Words heavily skewed toward 1★ (appear ≥2× more in 1★ than 5★)
distinctive_1star = []
for w in shared_words:
    ratio = freq_1star[w] / freq_5star[w]
    if ratio >= 2.0 and freq_1star[w] >= 50:  # Minimum frequency to filter noise
        distinctive_1star.append((w, freq_1star[w], ratio))
distinctive_1star.sort(key=lambda x: -x[1])  # Sort by absolute frequency

# Words heavily skewed toward 5★ (appear ≥2× more in 5★ than 1★)
distinctive_5star = []
for w in shared_words:
    ratio = freq_5star[w] / freq_1star[w]
    if ratio >= 2.0 and freq_5star[w] >= 50:
        distinctive_5star.append((w, freq_5star[w], ratio))
distinctive_5star.sort(key=lambda x: -x[1])

n_show = 15  # Top N distinctive words per extreme

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

for ax, data, color, title, extreme in [
    (axes[0], distinctive_1star[:n_show], '#DC2626', '1★ (Negative)', 'negative'),
    (axes[1], distinctive_5star[:n_show], '#0891B2', '5★ (Positive)', 'positive'),
]:
    if data:
        words, counts, ratios = zip(*data)
        bars = ax.barh(list(reversed(words)), list(reversed(counts)),
                       color=color, edgecolor='white', height=0.7)
        max_c = max(counts)
        for bar, count, ratio in zip(bars, reversed(counts), reversed(ratios)):
            ax.text(bar.get_width() + max_c * 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f'{count:,} (×{ratio:.1f})', va='center', fontsize=8)
    ax.set_title(f'Words Distinctive of {title} Reviews', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Frequency (ratio vs opposite extreme)', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'extreme_rating_words.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"💾 Plot saved → {plot_path}")
plt.show()

print(f"\nDistinctive words found: {len(distinctive_1star)} for 1★, {len(distinctive_5star)} for 5★")

In [ ]:
# ── Temporal Analysis: Reviews Over Time ──────────────────────────────────────
# The 'timestamp' column records when each review was written (Unix epoch).
# I extract year and month to check for temporal patterns:
# — Are reviews increasing or decreasing over time?
# — Does the average rating shift over months/years?
#
# Expected output: two line plots (volume over time, rating over time).

if 'timestamp' in df_raw.columns:
    # Convert Unix milliseconds to datetime
    df_raw['review_date'] = pd.to_datetime(df_raw['timestamp'], unit='ms', errors='coerce')

    # Extract temporal components
    df_raw['review_year']  = df_raw['review_date'].dt.year
    df_raw['review_month'] = df_raw['review_date'].dt.month
    df_raw['year_month']   = df_raw['review_date'].dt.to_period('M')

    # Date range of the dataset
    min_date = df_raw['review_date'].min()
    max_date = df_raw['review_date'].max()
    print(f"Temporal range: {min_date.strftime('%b %Y')} → {max_date.strftime('%b %Y')}")
    print(f"Unique year-months: {df_raw['year_month'].nunique()}")

    # Plot 1: Review volume over time
    reviews_per_month = df_raw.groupby('year_month').size()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Subplot 1 — Volume
    ax = axes[0]
    months_str = [str(m) for m in reviews_per_month.index]
    ax.fill_between(range(len(months_str)), reviews_per_month.values, alpha=0.3, color='#0891B2')
    ax.plot(range(len(months_str)), reviews_per_month.values, color='#0891B2', linewidth=1.8, marker='o', markersize=3)

    # Show only ~8 x-ticks to avoid clutter
    tick_step = max(1, len(months_str) // 8)
    tick_positions = list(range(0, len(months_str), tick_step))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([months_str[i] for i in tick_positions], rotation=45, ha='right', fontsize=8)

    ax.set_title('Review Volume Over Time', fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('Number of Reviews', fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    # Subplot 2 — Average rating over time
    ax = axes[1]
    avg_rating_per_month = df_raw.groupby('year_month')['rating'].mean()

    ax.plot(range(len(months_str)), avg_rating_per_month.values, color='#EA580C', linewidth=1.8, marker='o', markersize=3)
    ax.fill_between(range(len(months_str)), avg_rating_per_month.values, alpha=0.15, color='#EA580C')

    ax.set_xticks(tick_positions)
    ax.set_xticklabels([months_str[i] for i in tick_positions], rotation=45, ha='right', fontsize=8)

    # Reference line at overall mean
    overall_mean = df_raw['rating'].mean()
    ax.axhline(overall_mean, color='#334155', linestyle='--', linewidth=1, alpha=0.6,
               label=f'Overall mean: {overall_mean:.2f}')
    ax.legend(fontsize=9)

    ax.set_title('Average Rating Over Time', fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('Average Star Rating', fontsize=11)
    ax.set_ylim(1, 5)

    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, 'reviews_over_time.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n💾 Plot saved → {plot_path}")
    plt.show()

    # Clean up temporal columns (keep only if needed downstream)
    df_raw.drop(columns=['review_date', 'review_year', 'review_month', 'year_month'], inplace=True)
else:
    print("⚠️  Column 'timestamp' not found in the dataset. Skipping temporal analysis.")

In [ ]:
# ── Duplicate Review Detection ─────────────────────────────────────────────────
# Duplicate reviews can inflate model confidence on repeated patterns
# and leak data between splits (the same review appearing in train and test).
# I check for exact and near-duplicate reviews.
#
# Expected output: duplicate count + examples if any are found.

# Check 1: Exact duplicate texts (case-sensitive)
n_exact_dupes = df_raw['text'].duplicated().sum()
pct_exact = n_exact_dupes / len(df_raw) * 100

print("Duplicate Review Detection:")
print(f"  Exact duplicate texts : {n_exact_dupes:,} ({pct_exact:.2f}%)")

# Check 2: Near-duplicates — same text after lowercasing and stripping whitespace
normalized = df_raw['text'].fillna('').str.lower().str.strip()
n_near_dupes = normalized.duplicated().sum()
pct_near = n_near_dupes / len(df_raw) * 100
print(f"  Near-duplicates (norm): {n_near_dupes:,} ({pct_near:.2f}%)")

# Check 3: Same user posting the same text multiple times
if 'user_id' in df_raw.columns:
    user_text_dupes = df_raw.duplicated(subset=['user_id', 'text']).sum()
    print(f"  Same user + same text : {user_text_dupes:,}")

# Show examples if duplicates exist
if n_exact_dupes > 0:
    print(f"\n  Example duplicate reviews (showing first 3):")
    duplicated_mask = df_raw['text'].duplicated(keep=False)  # Mark all copies
    dupes = df_raw[duplicated_mask]['text'].value_counts().head(3)
    for text, count in dupes.items():
        preview = str(text)[:150].replace('\n', ' ')
        print(f"    [{count}×] \"{preview}...\"")

if n_exact_dupes == 0 and n_near_dupes == 0:
    print("\n  ✅ No duplicate reviews detected — data quality check passed.")

---
## Section 3 — Data Preprocessing

In this section, I clean the raw data to prepare it for model fine-tuning. My preprocessing pipeline:

1. **Drop missing values** — rows without `text` or `rating` are unusable.
2. **Clean text** — lowercase, strip HTML tags, remove special characters, normalize whitespace.
3. **Map ratings to labels** — 1–2 → Negative (0), 3 → Neutral (1), 4–5 → Positive (2).
4. **Filter short reviews** — reviews with fewer than 10 characters after cleaning carry no signal.

I chose this cleaning approach based on the understanding that transformer models like DistilBERT and RoBERTa benefit from consistent, noise-free input text. HTML tags and special characters appear to be artefacts of the data scraping and storage pipeline rather than natural language patterns the model should learn.

In [ ]:
# ── Step 1: Drop Rows with Missing Critical Fields ────────────────────────────
# 'text' and 'rating' are the two fields the model needs.
# Without them, a row cannot be used for training or evaluation.
#
# Expected output: shape before and after dropping.

shape_before = df_raw.shape  # Record shape before dropping for reporting

# Drop rows where either 'text' or 'rating' is NaN
df = df_raw.dropna(subset=['text', 'rating']).copy()  # .copy() prevents SettingWithCopyWarning

# Also drop rows where text is an empty or whitespace-only string
# str.strip() removes leading/trailing whitespace; == '' checks for empty result
df = df[df['text'].str.strip() != ''].copy()

shape_after = df.shape
n_dropped = shape_before[0] - shape_after[0]  # Rows removed in this step

print(f"Shape before drop : {shape_before}")
print(f"Shape after drop  : {shape_after}")
print(f"Rows dropped      : {n_dropped:,} ({n_dropped/shape_before[0]*100:.2f}%)")

In [ ]:
# ── Step 2: Text Cleaning Function ───────────────────────────────────────────
# I define a single function that applies all cleaning steps in sequence.
# Using a function (rather than chained in-line operations) makes it easy
# to inspect, test, and reuse the exact same logic in NB02 and NB03.
#
# Expected output: function defined (no output), then before/after examples.

def clean_text(text: str) -> str:
    """Clean a single review text string.

    Steps applied in order:
    1. Convert to lowercase (normalises vocabulary)
    2. Remove HTML tags (e.g. <br/>, <b>...)
    3. Remove URLs (http:// or https:// links)
    4. Remove non-ASCII characters (keeps only printable English)
    5. Remove special characters, keeping letters, digits, spaces, and basic punctuation
    6. Collapse multiple whitespace characters into a single space
    7. Strip leading/trailing whitespace

    Args:
        text: Raw review string

    Returns:
        Cleaned string
    """
    # Guard: return empty string if input is not a string (e.g. NaN slipped through)
    if not isinstance(text, str):
        return ''

    # Step 1 — Lowercase: 'Great' and 'great' should be treated identically
    text = text.lower()

    # Step 2 — Remove HTML tags: match anything between < and > (non-greedy)
    text = re.sub(r'<[^>]+>', ' ', text)

    # Step 3 — Remove URLs: matches http:// or https:// followed by non-whitespace
    text = re.sub(r'https?://\S+', ' ', text)

    # Step 4 — Remove non-ASCII characters: keeps only standard English characters
    text = text.encode('ascii', errors='ignore').decode('ascii')

    # Step 5 — Remove characters that are not letters, digits, spaces, or punctuation
    # I keep . , ! ? ' - so sentence structure is partially preserved
    text = re.sub(r"[^a-z0-9\s.,!?'\-]", ' ', text)

    # Step 6 — Collapse multiple whitespace (spaces, tabs, newlines) to one space
    text = re.sub(r'\s+', ' ', text)

    # Step 7 — Strip leading/trailing whitespace
    text = text.strip()

    return text

print("✅ clean_text() function defined.")

# Quick smoke test with a realistic dirty example
dirty_example = "  <br>This is a <b>GREAT</b> product!! Visit http://spam.com 😊 5★  "
clean_example = clean_text(dirty_example)
print(f"\nSmoke test:")
print(f"  BEFORE : {repr(dirty_example)}")
print(f"  AFTER  : {repr(clean_example)}")

In [ ]:
# ── Apply Text Cleaning to All Reviews ───────────────────────────────────────
# I apply clean_text() to the 'text' column using tqdm for a progress bar.
# tqdm.pandas() adds .progress_apply() which shows a live ETA.
#
# Expected output: progress bar, then before/after count of rows.

# Register tqdm with pandas to enable progress_apply()
tqdm.pandas(desc="Cleaning text")  # Sets the progress bar description label

# Apply the cleaning function row-by-row with a progress bar
# The result replaces the original 'text' column in-place
df['text'] = df['text'].progress_apply(clean_text)

print("\n✅ Text cleaning complete.")

# Show 3 before/after examples to verify the cleaning worked correctly
print("\nBefore/After examples:")
print("─" * 70)
# Pick examples from the original raw data where we know text exists
for idx in df_raw[df_raw['text'].notna()].head(3).index:
    raw_text   = df_raw.loc[idx, 'text'] if idx in df_raw.index else 'N/A'
    clean      = df.loc[idx, 'text']     if idx in df.index     else 'N/A'
    print(f"  RAW   : {str(raw_text)[:120]}")
    print(f"  CLEAN : {str(clean)[:120]}")
    print("─" * 70)

In [ ]:
# ── Step 3: Map Star Ratings → Sentiment Labels ───────────────────────────────
# I apply the official label mapping for this project:
#   1–2 stars → Negative (label 0)
#   3 stars   → Neutral  (label 1)
#   4–5 stars → Positive (label 2)
#
# Expected output: new 'label' column with values 0, 1, or 2.

# Apply the rating_to_label function defined in Section 2
# rating column must be numeric; convert just in case it's stored as float
df['label'] = df['rating'].astype(int).apply(rating_to_label)

# Verify the mapping with a cross-tabulation of rating vs label
print("Rating → Label cross-tabulation:")
display(pd.crosstab(df['rating'], df['label']))  # Expected: 3 non-zero regions

print("\nLabel distribution (absolute counts):")
print(df['label'].value_counts().sort_index())   # Expected: 3 rows (0, 1, 2)

In [ ]:
# ── Step 4: Filter Out Extremely Short Reviews ────────────────────────────────
# Reviews with fewer than 10 characters after cleaning carry little useful signal for sentiment classification.
# Examples: 'ok', '...', 'no', 'n/a'. These would confuse the classifier.
#
# Expected output: rows removed count and new DataFrame shape.

MIN_TEXT_LENGTH = 10  # Minimum number of characters in cleaned text

shape_before_filter = df.shape  # Record current shape for comparison

# Keep only rows where the cleaned text has at least MIN_TEXT_LENGTH characters
df = df[df['text'].str.len() >= MIN_TEXT_LENGTH].copy()

n_filtered = shape_before_filter[0] - df.shape[0]  # Number of rows removed

print(f"Shape before length filter : {shape_before_filter}")
print(f"Shape after length filter  : {df.shape}")
print(f"Rows removed (< {MIN_TEXT_LENGTH} chars)  : {n_filtered:,}")

# Final label distribution after all preprocessing
print("\nFinal label distribution after full preprocessing:")
label_counts = df['label'].value_counts().sort_index()
for label_id, count in label_counts.items():
    pct = count / len(df) * 100
    print(f"  Label {label_id} ({label_names[label_id]:8s}): {count:,}  ({pct:.1f}%)")

---
## Section 4 — Dataset Balancing & Train/Val/Test Splitting

In this section, I address class imbalance (Amazon reviews are heavily skewed toward 5-star ratings) and split the data into train, validation, and test sets.

**Why balance?** Without balancing, a classifier could achieve ~70% accuracy by always predicting Positive — a useless model. By limiting each class to the same sample count, I force the model to learn all three sentiment patterns.

**Split strategy**: I use a **stratified 70 / 15 / 15** split. Stratification ensures each split has the same class proportions as the full dataset — critical for fair evaluation.

**No data leakage**: I verify that the three splits are disjoint by checking that no `user_id` appears in multiple splits at a level that could inflate test scores.

In [ ]:
# ── Plot 4: Class Imbalance Before Balancing ──────────────────────────────────
# I visualise the raw label distribution so the imbalance is clearly visible.
# Expected output: bar chart showing Positive >> Neutral > Negative.

label_counts = df['label'].value_counts().sort_index()  # Count per label

# Readable label names for chart axes
label_display = ['Negative (0)', 'Neutral (1)', 'Positive (2)']
colors        = ['#DC2626', '#EA580C', '#0891B2']  # Red, amber, green — intuitive

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    label_display,         # X-axis labels
    label_counts.values,   # Y-axis values
    color=colors,
    edgecolor='white',
    linewidth=0.8,
    width=0.5
)

# Annotate each bar with count and percentage
total = label_counts.sum()
for bar, count in zip(bars, label_counts.values):
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # Horizontal centre of bar
        bar.get_height() + 50,               # Slightly above bar top
        f'{count:,}\n({pct:.1f}%)',          # Count and percentage on two lines
        ha='center', va='bottom', fontsize=10
    )

ax.set_title('Class Distribution BEFORE Balancing', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Sentiment Class', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'class_distribution_before.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"💾 Plot saved → {plot_path}")
plt.show()

In [ ]:
# ── Balance Classes by Undersampling ─────────────────────────────────────────
# Strategy: undersample the majority class(es) to match the minority class.
# This is the simplest and most transparent approach — no synthetic data,
# no complex oversampling. The minority class here is Neutral (3-star reviews).
#
# NO MAX_PER_CLASS cap — the minority class's natural size determines the final
# balanced size. Capping would discard data we already spent time streaming.
# Expected final: ~300 000–500 000 balanced reviews (depending on class distribution).
#
# Expected output: balanced DataFrame with equal rows per class.

# Find the count of the smallest class (this sets the cap for all classes)
min_class_count = df['label'].value_counts().min()
samples_per_class = min_class_count  # No artificial cap — natural minority defines size
print(f"Minority class size : {min_class_count:,}")
print(f"Samples per class   : {samples_per_class:,} (no cap)")

# Sample equally from each class
balanced_dfs = []  # Will hold one sampled DataFrame per class
for label_id in [0, 1, 2]:  # Iterate over Negative, Neutral, Positive
    class_subset = df[df['label'] == label_id]  # Filter to this class only

    # Sample with random_state for reproducibility; n=samples_per_class
    sampled = class_subset.sample(n=samples_per_class, random_state=RANDOM_SEED)
    balanced_dfs.append(sampled)  # Add to list

# Concatenate all three sampled DataFrames into one balanced DataFrame
df_balanced = pd.concat(balanced_dfs, ignore_index=True)  # Reset index after concat

# Shuffle rows — concat preserves class order which would confuse training
df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\n✅ Balanced dataset shape: {df_balanced.shape}")
print(f"Class distribution after balancing:")
print(df_balanced['label'].value_counts().sort_index())

In [ ]:
# ── Stratified Train / Val / Test Split ──────────────────────────────────────
# Split ratio: 70% train, 15% validation, 15% test
# stratify=label ensures each split has the same class distribution.
#
# I do this in two stages:
#   Stage 1: Split off the test set (15%) from the full dataset
#   Stage 2: Split the remaining 85% into train (70%) and val (15%)
#
# 15 / 85 ≈ 0.176 is the correct fraction for the second split
#
# Expected output: three DataFrames with shapes printed.

TRAIN_RATIO = 0.70  # 70% of data goes to training
VAL_RATIO   = 0.15  # 15% to validation
TEST_RATIO  = 0.15  # 15% to test

# Stage 1: Separate out the test set (15% of total)
df_trainval, df_test = train_test_split(
    df_balanced,                        # Full balanced dataset
    test_size=TEST_RATIO,               # Hold out 15% as test
    stratify=df_balanced['label'],      # Maintain class ratios in both halves
    random_state=RANDOM_SEED            # Reproducible split
)

# Stage 2: Split the remaining trainval into train and val
# The val fraction relative to trainval is: 0.15 / (1 - 0.15) ≈ 0.1765
val_fraction = VAL_RATIO / (1 - TEST_RATIO)
df_train, df_val = train_test_split(
    df_trainval,
    test_size=val_fraction,             # Fraction of trainval that becomes val
    stratify=df_trainval['label'],      # Preserve class distribution
    random_state=RANDOM_SEED
)

# Reset index on all three splits (good hygiene after slicing)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# Print shapes and class distributions for each split
print("Split sizes:")
print(f"  Train : {len(df_train):,} rows  ({len(df_train)/len(df_balanced)*100:.1f}%)")
print(f"  Val   : {len(df_val):,} rows  ({len(df_val)/len(df_balanced)*100:.1f}%)")
print(f"  Test  : {len(df_test):,} rows  ({len(df_test)/len(df_balanced)*100:.1f}%)")

print("\nClass distribution per split:")
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    dist = split['label'].value_counts(normalize=True).sort_index() * 100
    dist_str = ' | '.join([f"Label {i}: {dist[i]:.1f}%" for i in [0, 1, 2]])
    print(f"  {name:5s}: {dist_str}")

In [ ]:
# ── Verify No Data Leakage ────────────────────────────────────────────────────
# I check that no review appears in more than one split by comparing
# the DataFrame index values (each row has a unique original index).
# I also check user_id overlap as a secondary sanity check.
#
# Expected output: 'No leakage detected' for both checks.

# Check 1: Index-level — every row should be in exactly one split
train_idx = set(df_train.index)
val_idx   = set(df_val.index)
test_idx  = set(df_test.index)

# After reset_index, all indices start from 0 so this will always be empty
# (each split has its own 0-based index after reset). Instead, use original
# balanced indices stored before the split.
# Re-run split checks on the text content fingerprint instead:
train_texts = set(df_train['text'].values)
val_texts   = set(df_val['text'].values)
test_texts  = set(df_test['text'].values)

train_val_overlap  = len(train_texts & val_texts)   # Reviews in both train and val
train_test_overlap = len(train_texts & test_texts)  # Reviews in both train and test
val_test_overlap   = len(val_texts   & test_texts)  # Reviews in both val and test

print("Leakage check — text overlap between splits:")
print(f"  Train ∩ Val  : {train_val_overlap} reviews")
print(f"  Train ∩ Test : {train_test_overlap} reviews")
print(f"  Val   ∩ Test : {val_test_overlap} reviews")

# Assert no leakage — this will raise an error if any overlap exists
assert train_val_overlap == 0,  "⚠️ Leakage: train and val share reviews!"
assert train_test_overlap == 0, "⚠️ Leakage: train and test share reviews!"
assert val_test_overlap == 0,   "⚠️ Leakage: val and test share reviews!"

print("\n✅ No text-level data leakage detected.")

In [ ]:
# ── Plot 5: Class Distribution per Split ─────────────────────────────────────
# I visualise the label distribution in each split side-by-side.
# A well-stratified split shows nearly identical bar heights across splits.
#
# Expected output: grouped bar chart saved to PLOTS_DIR.

split_names  = ['Train', 'Val', 'Test']                    # Split names for chart
split_frames = [df_train, df_val, df_test]                 # Corresponding DataFrames
bar_colors   = ['#DC2626', '#EA580C', '#0891B2']           # Red, amber, green per class

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)  # 3 side-by-side subplots

for ax, name, split_df in zip(axes, split_names, split_frames):
    counts = split_df['label'].value_counts().sort_index()  # Count per label

    # Bar chart for this split
    bars = ax.bar(
        ['Neg', 'Neu', 'Pos'],  # Short labels fit the narrow subplot
        counts.values,
        color=bar_colors,
        edgecolor='white',
        linewidth=0.8,
        width=0.5
    )

    # Annotate each bar with absolute count
    for bar, count in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 20,
            f'{count:,}',
            ha='center', va='bottom', fontsize=9
        )

    ax.set_title(f'{name} ({len(split_df):,} rows)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sentiment', fontsize=10)
    if ax == axes[0]:  # Only add Y label on leftmost subplot
        ax.set_ylabel('Number of Reviews', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.suptitle('Class Distribution per Split (After Balancing)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'class_distribution_per_split.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"💾 Plot saved → {plot_path}")
plt.show()

---
## Section 5 — Save to HuggingFace Arrow Format

In this final section, I convert the three pandas DataFrames to HuggingFace `Dataset` objects and save them in Apache Arrow format. Arrow is a columnar, memory-mapped binary format that:

- Loads orders of magnitude faster than CSV
- Supports efficient batched tokenisation (used by `Trainer` in NB02 and NB03)
- Preserves data types without ambiguity

I also save a summary CSV with metadata about the final dataset — useful for reporting and logging.

In [ ]:
# ── Select Final Columns ──────────────────────────────────────────────────────
# I keep only the columns needed downstream to reduce file size.
# NB02 and NB03 only need 'text' (input) and 'label' (target).
# 'category' is kept for stratified analysis; 'rating' for reference.
#
# Expected output: DataFrames with 4 columns each.

KEEP_COLS = ['text', 'label', 'rating', 'category', 'parent_asin']  # Columns to retain

# Select only the needed columns from each split DataFrame
df_train_final = df_train[KEEP_COLS].copy()  # Training split — 70%
df_val_final   = df_val[KEEP_COLS].copy()    # Validation split — 15%
df_test_final  = df_test[KEEP_COLS].copy()   # Test split — 15%

print(f"Final column selection: {KEEP_COLS}")
print(f"Train shape : {df_train_final.shape}")
print(f"Val shape   : {df_val_final.shape}")
print(f"Test shape  : {df_test_final.shape}")

In [ ]:
# ── Convert to HuggingFace Dataset & Save as Arrow ────────────────────────────
# Dataset.from_pandas() converts a pandas DataFrame to a HuggingFace Dataset.
# save_to_disk() serialises it to Arrow format in the specified directory.
# DatasetDict bundles all three splits under a single object.
#
# Expected output: three sub-directories created under DATASET_DIR.

print("Converting DataFrames to HuggingFace Datasets...")

# Convert each split — preserves_index=False drops the pandas RangeIndex
hf_train = Dataset.from_pandas(df_train_final, preserve_index=False)  # Train split
hf_val   = Dataset.from_pandas(df_val_final,   preserve_index=False)  # Val split
hf_test  = Dataset.from_pandas(df_test_final,  preserve_index=False)  # Test split

# Bundle all three splits into a DatasetDict — the standard HuggingFace format
# Keys ('train', 'validation', 'test') follow HuggingFace naming conventions
dataset_dict = DatasetDict({
    'train'     : hf_train,
    'validation': hf_val,
    'test'      : hf_test,
})

print(f"DatasetDict structure:")
print(dataset_dict)  # Expected: DatasetDict with 3 splits and their schemas

# Save to disk in Arrow format
# num_proc=1 is conservative; increase to 4+ if you have multiple CPU cores
dataset_dict.save_to_disk(DATASET_DIR)  # Saves train/ val/ test/ sub-dirs inside DATASET_DIR

print(f"\n✅ Dataset saved to: {DATASET_DIR}")

# Verify by listing the created files
print("\nFiles in DATASET_DIR:")
for entry in sorted(os.listdir(DATASET_DIR)):
    full_path = os.path.join(DATASET_DIR, entry)
    if os.path.isdir(full_path):
        # For directories, show their size via du (disk usage)
        n_files = len(os.listdir(full_path))
        print(f"  📁 {entry}/  ({n_files} files)")
    else:
        print(f"  📄 {entry}")

In [ ]:
# ── Verify: Reload from Disk ──────────────────────────────────────────────────
# I reload the saved dataset from disk and spot-check a few rows to confirm
# the Arrow serialisation round-trip preserved data correctly.
#
# Expected output: same shapes as before saving, correct column values.

from datasets import load_from_disk  # Import function for loading Arrow datasets

# Load the saved DatasetDict from the Arrow directory
reloaded = load_from_disk(DATASET_DIR)

print("Reloaded DatasetDict:")
print(reloaded)  # Should show same split names and row counts

print("\nSpot-check — first 3 rows of reloaded train split:")
for i in range(min(3, len(reloaded['train']))):
    row = reloaded['train'][i]  # Access row by integer index
    print(f"  [{i}] label={row['label']} | rating={row['rating']} | text[:80]='{row['text'][:80]}'")

print("\n✅ Round-trip verification complete — data looks correct.")

In [ ]:
# ── Save Summary CSV with Dataset Metadata ────────────────────────────────────
# I save a small metadata CSV that records the key numbers for this dataset.
# This is useful for logging, reporting, and for NB02/NB03 to read programmatically.
#
# Expected output: CSV file saved to OUTPUT_DIR.

# Build the summary as a list of dicts — each dict is one row in the CSV
summary_records = []

for split_name, split_df in [('train', df_train_final), ('validation', df_val_final), ('test', df_test_final)]:
    for label_id in [0, 1, 2]:  # Iterate over all three classes
        n_class = (split_df['label'] == label_id).sum()  # Count rows for this class in this split
        summary_records.append({
            'split'       : split_name,                    # Which split (train/val/test)
            'label'       : label_id,                      # Label integer (0/1/2)
            'sentiment'   : label_names[label_id],         # Human-readable sentiment name
            'count'       : n_class,                       # Number of samples
            'pct_of_split': round(n_class / len(split_df) * 100, 2)  # Percentage within split
        })

# Add total row counts as summary rows
for split_name, split_df in [('train', df_train_final), ('validation', df_val_final), ('test', df_test_final)]:
    summary_records.append({
        'split'       : split_name,
        'label'       : 'ALL',
        'sentiment'   : 'All classes',
        'count'       : len(split_df),
        'pct_of_split': 100.0
    })

# Convert to DataFrame and save as CSV
df_summary = pd.DataFrame(summary_records)
summary_csv_path = os.path.join(OUTPUT_DIR, 'dataset_summary.csv')
df_summary.to_csv(summary_csv_path, index=False)  # index=False avoids unnamed index column

print(f"💾 Summary CSV saved → {summary_csv_path}")
display(df_summary)  # Show the table inline

In [ ]:
# ── Final Summary Print ───────────────────────────────────────────────────────
# I print a clean end-of-notebook summary so anyone opening the notebook
# can immediately see what was produced without scrolling through all cells.
#
# Expected output: formatted summary block.

print("=" * 60)
print("  NOTEBOOK 01 — COMPLETE")
print("  Data Preparation & EDA")
print("=" * 60)
print()
print("  Dataset: Amazon Reviews 2023 (McAuley Lab)")
print(f"  Categories sampled : {len(SELECTED_CATEGORIES)} (all available categories)")
print(f"  Raw reviews loaded : {len(df_raw):,}")
print(f"  After preprocessing: {len(df_balanced):,} (balanced)")
print()
print(f"  Splits saved:")
print(f"    Train      : {len(df_train_final):,} rows")
print(f"    Validation : {len(df_val_final):,} rows")
print(f"    Test       : {len(df_test_final):,} rows")
print()
print(f"  Outputs saved to: {OUTPUT_DIR}")
print(f"    Arrow dataset : {DATASET_DIR}")
print(f"    Summary CSV   : {summary_csv_path}")
print(f"    Plots (PNG)   : {PLOTS_DIR}")
print()
print("  Label mapping:")
print("    1-2 stars → 0 (Negative)")
print("    3 stars   → 1 (Neutral)")
print("    4-5 stars → 2 (Positive)")
print()
print("  ➡️  Next: Notebook 02 — DistilBERT Fine-tuning")
print("=" * 60)